# 1. Wallet Tagging — ML Classification Study

**Goal**: Train and evaluate models to classify blockchain wallets into categories:
- `exchange` (Binance, Coinbase, etc.)
- `mixer` (Tornado Cash, Railgun)
- `bridge` (Stargate, Wormhole)
- `smart_contract` / `defi`
- `bot` (MEV, trading bots)
- `bad_wallet` (attacker, drainer)
- `normal` (regular user)

**Pipeline**: PostgreSQL → Feature Engineering → Train → MLflow → SHAP → Evaluate

**Stack**: scikit-learn, XGBoost, LightGBM, SHAP, Plotly, MLflow

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from notebooks.src.data_loader import DataLoader
from notebooks.src.model_visualizer import ModelVisualizer as mviz

loader = DataLoader()
print('Connected to database')

## 1.1 Load Known Entities (Ground Truth)

In [ ]:
# Load known mixers, bridges, exchanges from DB
known_entities = loader.get_known_entities()
print(f'Known entities: {len(known_entities)}')
known_entities.groupby('type').size()

## 1.2 Load Wallet Scores (Existing Classifications)

In [ ]:
scores_df = loader.get_wallet_scores()
print(f'Total scored wallets: {len(scores_df)}')
if not scores_df.empty:
    display(scores_df.groupby('predicted_type').agg(
        count=('address', 'size'),
        avg_confidence=('confidence', 'mean'),
        avg_tx_count=('tx_count', 'mean'),
    ).sort_values('count', ascending=False))

## 1.3 Feature Engineering

Extract 50+ features per wallet using `WalletFeatureEngineer`.

In [ ]:
from api.services.feature_engineer import WalletFeatureEngineer

# Create feature engineer with DB session
session = loader.session
engineer = WalletFeatureEngineer(session=session)
print(f'Feature categories: {list(engineer.FEATURE_CATEGORIES.keys())}')
print(f'Total features: {len(engineer.get_feature_names())}')

In [ ]:
# Extract features for known entities (supervised training data)
# NOTE: This queries per-token transfer tables — only works for tokens we've synced
from api.application.erc20models import CHAIN_ID_TO_TRIGRAM

features_list = []
labels = []

for _, entity in known_entities.iterrows():
    try:
        feats = engineer.extract_features(
            address=entity['address'],
            chain='ETH',  # Default chain
            lookback_days=180
        )
        if feats and any(v != 0 for v in feats.values()):
            features_list.append(feats)
            labels.append(entity['type'])
    except Exception as e:
        continue

print(f'Extracted features for {len(features_list)} / {len(known_entities)} entities')

if features_list:
    X_df = pd.DataFrame(features_list).fillna(0)
    y = pd.Series(labels)
    print(f'Feature matrix: {X_df.shape}')
    print(f'Label distribution:\n{y.value_counts()}')

## 1.4 Train Multiple Models

Following mission7 pattern: DummyClassifier → LogisticRegression → RandomForest → XGBoost → LightGBM

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y) if len(features_list) > 0 else np.array([])

# Define models
models = {
    'Dummy (baseline)': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': LogisticRegression(max_iter=1000, multi_class='auto'),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
}

# Cross-validate each
cv = StratifiedKFold(n_splits=min(5, len(y_encoded) // 3 + 1), shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
    try:
        scores = cross_val_score(pipe, X_df, y_encoded, cv=cv, scoring='f1_weighted')
        results[name] = {'f1_weighted': scores.mean(), 'f1_std': scores.std()}
        print(f'{name}: F1={scores.mean():.3f} ± {scores.std():.3f}')
    except Exception as e:
        print(f'{name}: FAILED — {e}')
        results[name] = {'f1_weighted': 0, 'f1_std': 0}

## 1.5 SHAP Feature Importance

In [ ]:
# Train best model on full data for SHAP
if len(features_list) > 10:
    best_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', RandomForestClassifier(n_estimators=100, random_state=42))
    ])
    best_pipe.fit(X_df, y_encoded)
    
    importances = best_pipe.named_steps['model'].feature_importances_
    fig = mviz.plot_feature_importance(importances, X_df.columns.tolist(), top_n=20)
    fig.show()
else:
    print('Not enough data for SHAP analysis. Need more labeled wallets.')
    print('LIMITATION: Requires synced transfer data for known entities.')
    print('TAG: manual_labeling_needed')

## 1.6 Limitations & Tags

| Limitation | Impact | Mitigation |
|:-----------|:-------|:-----------|
| Small labeled dataset | Low accuracy for rare types (mixer, bridge) | Import more labels from Etherscan API |
| Only per-token tables | Can't classify wallets without synced tokens | Use InvestigationTransfer data as fallback |
| Heuristic thresholds | Hard-coded, may not generalize | Train threshold from data, use Optuna |
| No smart contract detection | Missing bytecode analysis | Etherscan `getabi` API call needed |

**TAG: `aria_manual_override`** — For wallets where ML confidence < 0.6, flag for Aria manual review.